# Lab 16 · Deploy a Foundry hosted agent behind AI Gateway

Deploy a Microsoft Agent Framework agent to Foundry Hosted Agents with azd. The agent has no model deployment, no
direct tool access, and no keys: it reaches its model and its Microsoft Learn tools only through your AI Gateway,
which authorizes its Microsoft Entra identities with an Azure role. This notebook is the code track of the
[lab guide](README.md): it runs the guide's azd project in [infra/](infra/) unchanged, in an azd environment of its
own, then caps the agent's tokens without touching its code.

**Provision the project → Grant AI Gateway User → Deploy the agent → Run it locally and in Foundry → Cap its tokens**

## Architecture at a glance

| Component | What it does |
| --- | --- |
| [infra/azure.yaml](infra/azure.yaml) | The azd project: a Foundry project with no model deployments, the `gateway-mcp` connection, the `gateway-tools` toolbox, and the hosted agent `gateway-agent`. |
| [infra/src/gateway-agent/main.py](infra/src/gateway-agent/main.py) | The agent: a model client that calls the gateway with a Microsoft Entra token, and the Foundry toolbox. |
| Model endpoint `<gateway-url>/default/models/openai/v1` | The agent sends the alias from lab 02 as its model, with a token of its own Agent ID in Foundry, or of you locally. |
| MCP endpoint `<gateway-url>/default/toolservers/learn/mcp` | The `learn` MCP server from lab 04. The toolbox reaches it through the connection, which signs in with the project's managed identity. |
| Role **AI Gateway User** | Lets an identity invoke models and call MCP tools. The agent, the project, and you each get it on the gateway. |
| azd environment `aigw-lab16-nb` | Names the resource group and the Foundry project. It's separate from the guide's `aigw-lab16`, so both can exist. |
| [../shared/utils.py](../shared/utils.py) | Helpers that every lab notebook shares: settings, your `az login` sign-in, ARM calls, and role assignments. |

![A Foundry hosted agent calls a model alias and the Learn MCP server through AI Gateway](../assets/diagrams/hosted-agent.svg)

## Prerequisites

- The [Azure CLI](https://learn.microsoft.com/cli/azure/install-azure-cli), signed in with `az login`, with
  permission to create a resource group and a Foundry resource, and to create role assignments on the gateway, for
  example as **Owner**. **Contributor** can't grant roles.
- The [Azure Developer CLI](https://learn.microsoft.com/azure/developer/azure-developer-cli/install-azd) (azd)
  1.37.0 or later, signed in with `azd auth login` to the same tenant. With
  [uv](https://docs.astral.sh/uv/getting-started/installation/) installed, azd uses it to create the agent's virtual
  environment. Otherwise, it needs Python 3.13 or later.
- Python 3.10 or later with Jupyter, for example VS Code with the Python and Jupyter extensions.
- [02-first-model-call.ipynb](../02-first-model-call/02-first-model-call.ipynb) has run, so the `gpt-5-mini` alias
  exists, and [04-federate-mcp-tools.ipynb](../04-federate-mcp-tools/04-federate-mcp-tools.ipynb) has run with
  `keep_for_lab_05 = True`, so the `learn` MCP server exists.
- Port 8088 is free, for the local run.

> **Cost:** azd creates a Foundry resource, and the hosted agent and its model calls are billable. Run
> **Clean up resources** when you're done. The whole notebook takes about 20 minutes.

Run the cells in order, or click **Run All**. Each step is idempotent: rerun any cell and it reuses what exists.

In [ ]:
%pip install -q --upgrade azure-identity requests

### 0. Initialize the lab

The defaults match notebooks 01, 02, and 04, so `utils.Lab` finds the gateway without edits. The cell checks that the
alias and the MCP server exist, and that azd is recent enough and signed in. If `pip` upgraded a package that the
kernel had already loaded, restart the kernel before you run this cell.

In [ ]:
import json
import os
import re
import shutil
import signal
import socket
import subprocess
import sys
import tempfile
import time
import uuid
from pathlib import Path

from IPython.display import display

sys.path.insert(0, "../shared")   # helpers and brand styles that every lab notebook shares
import brand
import utils

lab = utils.Lab("16-foundry-hosted-agent")
alias       = "gpt-5-mini"      # model alias from lab 02, which the agent sends as its model
server_name = "learn"           # MCP server from lab 04, which the toolbox calls
env_name    = "aigw-lab16-nb"   # the azd environment, which names the resource group and the Foundry project
location    = "eastus2"         # a region with Foundry Hosted Agents
question    = ("Find the Microsoft Learn article that explains Azure API Management key concepts. "
               "Reply with its title and URL in one sentence.")
infra = Path("infra").resolve()

alias_resource = next((a for a in lab.arm("GET", f"{lab.workspace_id}/aliases")["value"]
                       if a["properties"].get("aliasName", a["name"]) == alias), None)
assert alias_resource, f"Alias {alias} not found. Run notebook 02 first."
lab.require(f"{lab.workspace_id}/toolServers/{server_name}", "Run notebook 04 with keep_for_lab_05 = True first.")

azd_path = shutil.which("azd")
assert azd_path, "Install the Azure Developer CLI from https://aka.ms/azd-install, then restart the kernel."
version = subprocess.run([azd_path, "version"], capture_output=True, text=True).stdout
version = re.search(r"(\d+)\.(\d+)\.(\d+)", version)
assert version and tuple(map(int, version.groups())) >= (1, 37, 0), "Upgrade azd to 1.37.0 or later."
status = subprocess.run([azd_path, "auth", "login", "--check-status"], capture_output=True, text=True)
assert status.returncode == 0 and "Logged in" in status.stdout, "Run `azd auth login`, then rerun the cell."

display(brand.info("What the agent uses", [
    ("Alias", brand.mono(alias)),
    ("MCP server", brand.mono(server_name)),
    ("azd", f"{version[0]}, {status.stdout.strip()}"),
    ("azd environment", brand.mono(env_name)),
    ("Foundry region", location),
]))

### 1. Configure the azd project

`azd` runs one azd command in `infra/` for the notebook's environment, and shows its latest line while it runs. The
cell installs the Foundry extensions for azd, creates the `aigw-lab16-nb` environment in your gateway's subscription,
or reuses it, and stores the three values that [azure.yaml](infra/azure.yaml) reads. None of them is a secret. If
`infra/` already has a default azd environment, such as the guide's `aigw-lab16`, it stays the default.

In [ ]:
ANSI = re.compile(r"\x1b\[[0-?]*[ -/]*[@-~]")


def azd(*args, title=None, status=None, environment=True, check=True, timeout=1800):
    """Run azd in infra/ and return (exit code, output, seconds).

    With a title, show the latest line in status, or in a new output line.
    """
    command = [azd_path, *args, *(["-e", env_name] if environment else []), "--no-prompt"]
    if title and status is None:
        status = brand.live()
    start, lines = time.time(), []
    process = subprocess.Popen(command, cwd=infra, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
                               encoding="utf-8", errors="replace", env={**os.environ, "NO_COLOR": "1"})
    for line in process.stdout:
        line = ANSI.sub("", line).rstrip()
        if line.strip():
            lines.append(line)
            if title:
                status.update(brand.note(f"{title} · {round(time.time() - start)} s", "wait", code=line.strip()[:110]))
    exit_code = process.wait(timeout=timeout)
    seconds = round(time.time() - start)
    if title:
        status.update(brand.note(f"{title} · {seconds} s", "ok" if exit_code == 0 else "warn"))
    if check:
        assert exit_code == 0, f"azd {' '.join(args[:3])} failed:\n" + "\n".join(lines[-30:])
    return exit_code, "\n".join(lines), seconds


def env_values():
    """Read the azd environment's values: none of them is a secret."""
    return json.loads(azd("env", "get-values", "--output", "json")[1])


status = brand.live()
azd("ext", "install", "microsoft.foundry", title="Install the Foundry extensions for azd", status=status,
    environment=False)

config = infra / ".azure" / "config.json"
created_azure_folder = not config.parent.exists()
created_env = not (infra / ".azure" / env_name).exists()
if created_env:
    saved_config = config.read_text() if config.exists() else None
    azd("env", "new", env_name, "--location", location, "--subscription", lab.subscription_id,
        title=f"Create the azd environment {env_name}", status=status, environment=False)
    if saved_config:
        config.write_text(saved_config)   # keep the folder's default environment

settings = {"AI_GATEWAY_URL": lab.gateway_url, "AI_GATEWAY_MODEL": alias, "AI_GATEWAY_MCP_SERVER": server_name}
for name, value in settings.items():
    azd("env", "set", name, value, title=f"Set {name}", status=status)
status.update(brand.note(f"{'Created' if created_env else 'Reusing'} the azd environment {env_name}", "ok"))
display(brand.table(f"azd environment {env_name}", ["Value", "Setting", "Used by"], [
    (brand.mono("AI_GATEWAY_URL"), brand.mono(lab.gateway_url), "The connection and the agent"),
    (brand.mono("AI_GATEWAY_MODEL"), brand.mono(alias), "The agent, as its model"),
    (brand.mono("AI_GATEWAY_MCP_SERVER"), brand.mono(server_name), "The connection's target"),
], sub=f"Subscription {lab.subscription_id} · {location}"))

### 2. Provision the Foundry project

`azd provision` creates the resource group, the Foundry resource, and a project with hosted agents enabled, and no
model deployments. It takes about two minutes. The cell then reads the project's managed identity, which calls the
MCP endpoint through the connection.

In [ ]:
azd("provision", title="Provision the Foundry project")

values = env_values()
project_id = values["AZURE_AI_PROJECT_ID"]
account_id = project_id.split("/projects/")[0]
account_name, project_name = account_id.rsplit("/", 1)[1], project_id.rsplit("/", 1)[1]
project_principal_id = lab.arm("GET", project_id, api_version=utils.COGNITIVE_API)["identity"]["principalId"]
display(brand.info("Foundry project", [
    ("Resource group", brand.mono(account_id.split("/")[4])),
    ("Foundry resource", brand.mono(account_name)),
    ("Project", brand.mono(project_name)),
    ("Project's managed identity", brand.mono(project_principal_id)),
], badge=brand.chip("Provisioned", "ok")))

### 3. Give the project and yourself access

Three identities call the gateway, and each one needs **AI Gateway User**. The project's managed identity calls the
MCP endpoint, and you call the model endpoint when the agent runs locally. The agent's own identity exists only after
the first deploy, so it gets the role in step 6.

`grant` assigns the role on the gateway, unless the identity has it already. The assignment's name derives from the
gateway and the identity, so **Clean up** can tell your role from one you had before. A brand-new identity can take a
minute to reach Microsoft Entra ID, so `grant` retries on `PrincipalNotFound`.

In [ ]:
GATEWAY_USER = "caad7a21-134e-43f4-a550-c5ffc17e051c"   # AI Gateway User: invoke models and call MCP tools


def assignment_path(principal_id):
    """The role assignment this notebook creates for the identity, named after the gateway and the identity."""
    name = uuid.uuid5(uuid.NAMESPACE_URL, f"{lab.gateway_id}/{principal_id}/{GATEWAY_USER}")
    return f"{lab.gateway_id}/providers/Microsoft.Authorization/roleAssignments/{name}"


def has_gateway_user(principal_id):
    return any(a["role"] == "AI Gateway User" and lab.gateway_id.lower().startswith(a["scope"].lower())
               for a in lab.role_assignments(principal_id))


def grant(principal_id, principal_type):
    """Assign AI Gateway User on the gateway, unless the identity has it. Return True if this call assigned it."""
    if has_gateway_user(principal_id):
        return False
    body = {"properties": {"principalId": principal_id, "principalType": principal_type, "roleDefinitionId":
            f"/subscriptions/{lab.subscription_id}/providers/Microsoft.Authorization/roleDefinitions/{GATEWAY_USER}"}}
    deadline = time.time() + 300
    while True:
        try:
            lab.arm("PUT", assignment_path(principal_id), body, api_version=utils.ROLES_API)
            return True
        except RuntimeError as error:
            if "RoleAssignmentExists" in str(error):
                return False
            if "PrincipalNotFound" not in str(error) or time.time() > deadline:
                raise
            time.sleep(15)


def role_chip(assigned):
    return brand.chip("Assigned", "ok") if assigned else brand.chip("Already had it", "neutral")


my_object_id, my_type, my_name = lab.signed_in_identity()
identities = [   # (who, shown in Access > Entra ID as, object ID, principal type, what it calls)
    ("You", my_name, my_object_id, my_type, "The model endpoint, when the agent runs locally"),
    ("Foundry project", f"{account_name}/projects/{project_name}", project_principal_id, "ServicePrincipal",
     "The MCP endpoint, through the connection"),
]
display(brand.table("AI Gateway User on the gateway", ["Identity", "Shown in Access › Entra ID as", "Calls", "Role"], [
    (who, brand.mono(shown), calls, role_chip(grant(object_id, principal_type)))
    for who, shown, object_id, principal_type, calls in identities
], sub="Round 1 of 2: the agent gets its role once it's deployed"))

### 4. Read the agent

The agent is about 80 lines. Its model client points at the gateway's model endpoint, sends the alias as its model,
and sends a Microsoft Entra token for the gateway's audience instead of a key. In Foundry, the token belongs to the
agent's own identity; locally, to you. `FoundryToolbox` calls the `gateway-tools` toolbox, which reaches the gateway's
MCP endpoint through the `gateway-mcp` connection, so the agent never holds a credential for the MCP server.

In [ ]:
source = (infra / "src" / "gateway-agent" / "main.py").read_text()
display(brand.card("src/gateway-agent/main.py", brand.pre(source),
                   sub=f"{len(source.splitlines())} lines · Microsoft Agent Framework"))
review = [
    ("Calls the model endpoint under /default/", "/default/models/openai/v1/" in source),
    ("Gets a Microsoft Entra token for the gateway's audience", "https://azure-api.net/aigateway/.default" in source),
    ("Reaches its tools only through the Foundry toolbox", "FoundryToolbox" in source),
    ("Reads no key from the environment", "API_KEY" not in source),
]
display(brand.table("What the agent does", ["Check", "Result"], [
    (label, brand.chip("Yes", "ok") if ok else brand.chip("No", "bad")) for label, ok in review]))

### 5. Deploy to Foundry

`azd deploy --all` creates the `gateway-mcp` connection, gives you **Foundry User** on the project for the toolbox,
creates the `gateway-tools` toolbox, uploads `src/gateway-agent` as a code package, and creates the hosted agent.
Foundry builds it from `requirements.txt` and gives it its own identity. It takes about three minutes. The first
deploy can show `AZURE_CONTAINER_REGISTRY_ENDPOINT not set; skipping ACR role check`: the agent deploys as code, so
that's expected.

In [ ]:
_, output, seconds = azd("deploy", "--all", title="Deploy the connection, the toolbox, and the agent")

values = env_values()
agent_principal_id = values["AGENT_GATEWAY_AGENT_INSTANCE_IDENTITY_PRINCIPAL_ID"]
display(brand.info("Hosted agent gateway-agent", [
    ("Version", values.get("AGENT_GATEWAY_AGENT_VERSION", "")),
    ("Agent identity", brand.mono(agent_principal_id)),
    ("Responses endpoint", brand.mono(values.get("AGENT_GATEWAY_AGENT_RESPONSES_ENDPOINT", ""))),
    ("Toolbox endpoint", brand.mono(values.get("TOOLBOX_GATEWAY_TOOLS_MCP_ENDPOINT", ""))),
], sub=f"Deployed in {seconds} s", badge=brand.chip("Deployed", "ok")))

### 6. Give the agent access

Foundry created the agent's identity, a Microsoft Entra Agent ID, on deploy. It stays the same when you deploy new
versions, so you grant it once. The role takes a couple of minutes to apply: the next step runs the agent locally in
the meantime.

In [ ]:
identities.append(("Hosted agent", f"{account_name}-{project_name}-gateway-agent-AgentIdentity", agent_principal_id,
                   "ServicePrincipal", "The model endpoint, from Foundry"))
who, shown, object_id, principal_type, calls = identities[-1]
display(brand.table("AI Gateway User on the gateway", ["Identity", "Shown in Access › Entra ID as", "Calls", "Role"], [
    (who, brand.mono(shown), calls, role_chip(grant(object_id, principal_type)))], sub="Round 2 of 2"))

### 7. Run the agent locally

`azd ai agent run --no-client` creates a virtual environment in `src/gateway-agent`, installs `requirements.txt`,
and starts the agent on `http://localhost:8088` with the azd environment's values. The first start takes a minute or
two. The cell then asks the question with `azd ai agent invoke --local`, and stops the agent, as **Ctrl+C** would.

Locally, the agent calls the model as you, and the Learn tools through the toolbox in Foundry. Behind the answer are
a model call that asks for a tool, one or more Learn tool calls, and a model call that writes the answer, all through
the gateway. When an attempt fails, `diagnose` names the last error in the agent's log, such as a `403` until your
role applies, and the cell asks again in a new conversation.

In [ ]:
venv = infra / "src" / "gateway-agent" / ".venv"
created_venv = not venv.exists()
log_path = Path(tempfile.gettempdir()) / f"{env_name}-agent-run.log"
ERRORS = [   # the patterns of the errors this lab can hit, with what they mean
    (r"Error code: (\d{3})", lambda m: f"HTTP {m[1]} from the gateway"
     + (": the role may not have applied yet" if m[1] == "403" else "")),
    (r"Access denied when connecting to the MCP server.*?\(HTTP (\d{3})",
     lambda m: f"HTTP {m[1]} on a tool call through the toolbox"),
    (r"Token limit is exceeded", lambda m: "HTTP 429 from the gateway's token limit"),
    (r"exceeded token rate limit",
     lambda m: "HTTP 429: the model deployment behind the alias ran out of tokens per minute"),
]


def diagnose(text):
    """Explain the last error in an agent's log or output, or return None."""
    found = [(m.start(), explain(m)) for pattern, explain in ERRORS for m in re.finditer(pattern, text, re.I)]
    return max(found)[1] if found else None


ERROR_LINE = re.compile(r"Error code: \d|Access denied when|token rate limit|Token limit is exceeded|^ERROR: "
                        r"|\w+Error: ")


def error_lines(text, count=3):
    """The last distinct error lines, without the agent's telemetry."""
    lines = [line.strip()[:300] for line in text.splitlines()
             if ERROR_LINE.search(line.strip()) and not line.strip().startswith('"')]   # skips telemetry JSON
    return "\n".join(list(dict.fromkeys(reversed(lines)))[:count][::-1])


def log_since(offset):
    with open(log_path, encoding="utf-8", errors="replace") as log:
        log.seek(offset)
        return log.read()


def port_open():
    try:
        socket.create_connection(("localhost", 8088), timeout=2).close()
        return True
    except OSError:
        return False


def answer_of(output, target):
    """The agent's answer in azd ai agent invoke's output, after [local] or [<agent name>], without azd's timings."""
    found = re.search(rf"^\[{re.escape(target)}\]\s*(.+?)(?=^\s*Client elapsed:|\Z)", output, re.M | re.S)
    return found[1].strip() if found else ""


def cites(answer):
    if "learn.microsoft.com" in answer:
        return brand.chip("Cites Microsoft Learn", "ok")
    return brand.chip("No Learn URL", "warn")


def stop(process):
    """Stop azd and the agent it started, as Ctrl+C would."""
    if process.poll() is None:
        if os.name == "nt":
            subprocess.run(["taskkill", "/T", "/F", "/PID", str(process.pid)], capture_output=True)
        else:
            os.killpg(process.pid, signal.SIGINT)
            try:
                process.wait(timeout=30)
            except subprocess.TimeoutExpired:
                os.killpg(process.pid, signal.SIGKILL)
    process.wait()


assert not port_open(), "Port 8088 is in use. Stop the agent that runs there, then rerun the cell."
group = {"creationflags": subprocess.CREATE_NEW_PROCESS_GROUP} if os.name == "nt" else {"start_new_session": True}
with open(log_path, "w") as log:
    server = subprocess.Popen([azd_path, "ai", "agent", "run", "--no-client", "-e", env_name, "--no-prompt"],
                              cwd=infra, stdout=log, stderr=subprocess.STDOUT, **group)
local_answer, output = "", ""
try:
    status, start = brand.live(), time.time()
    while not port_open():
        assert server.poll() is None, "The agent stopped:\n" + log_path.read_text()[-3000:]
        assert time.time() - start < 600, "The agent didn't start in 10 minutes:\n" + log_path.read_text()[-3000:]
        status.update(brand.note(f"Starting the agent on localhost:8088 · {round(time.time() - start)} s", "wait"))
        time.sleep(5)
    status.update(brand.note(f"The agent listens on localhost:8088 · started in {round(time.time() - start)} s", "ok"))
    status, flags, reason = brand.live(), [], None
    for attempt in range(10):   # your role can take a couple of minutes to apply
        offset = log_path.stat().st_size
        exit_code, output, seconds = azd("ai", "agent", "invoke", "--local", question, *flags,
                                         title="Ask the local agent", status=status, check=False)
        local_answer = answer_of(output, "local")
        reason = diagnose(log_since(offset) + "\n" + output)
        if exit_code == 0 and local_answer and ("learn.microsoft.com" in local_answer or not reason):
            break
        pause = 60 if "per minute" in (reason or "") else 30
        status.update(brand.note(f"{reason or 'The agent did not answer'} · asking again in {pause} s", "wait"))
        flags = ["--new-conversation"]
        time.sleep(pause)
finally:
    stop(server)
assert local_answer, f"The local agent didn't answer. {reason or ''}\n" + error_lines(log_path.read_text() + output)
display(brand.reply(local_answer, "Local agent", model=alias, sub=f"azd ai agent invoke --local · {seconds} s",
                    badge=cites(local_answer), markdown=True))
if reason and "learn.microsoft.com" not in local_answer:
    display(brand.note(reason, "warn"))

### 8. Invoke the hosted agent

Ask the hosted agent the same question in Foundry, where it calls the model with its own identity. The first call
starts a new session, which can take 30 seconds or more.

When a hosted call fails, azd shows only `ERROR: agent failed (server_error)`. `agent_logs` reads the agent's last
300 log lines with `azd ai agent monitor`, and `diagnose` names the last error in them. Until the agent's role
applies, it's a `403` from the gateway, and the cell asks again every 30 seconds, for up to 10 minutes.

In [ ]:
def agent_logs():
    """The hosted agent's last 300 log lines."""
    return azd("ai", "agent", "monitor", "gateway-agent", "--tail", "300", check=False)[1]


def gateway_errors():
    """The gateway's errors in the hosted agent's latest logs, as (status, message) pairs."""
    return re.findall(r"Error code: (\d{3}) - .*?'message': '([^']*)'", agent_logs())


status, deadline, flag = brand.live(), time.time() + 600, "--new-session"
while True:
    exit_code, output, seconds = azd("ai", "agent", "invoke", "gateway-agent", question, flag,
                                     title="Ask the hosted agent", status=status, check=False)
    hosted_answer = answer_of(output, "gateway-agent")
    if exit_code == 0 and hosted_answer:
        break
    logs = agent_logs()
    reason = diagnose(logs) or "The agent failed"
    assert time.time() < deadline, f"The hosted agent didn't answer. {reason}\n" + error_lines(logs + "\n" + output)
    pause = 60 if "per minute" in reason else 30
    status.update(brand.note(f"{reason} · asking again in {pause} s", "wait"))
    flag = "--new-conversation"
    time.sleep(pause)
display(brand.reply(hosted_answer, "Hosted agent gateway-agent", model=alias,
                    sub=f"azd ai agent invoke gateway-agent · {seconds} s", badge=cites(hosted_answer), markdown=True))
if "learn.microsoft.com" not in hosted_answer:
    display(brand.note(diagnose(agent_logs()) or "The agent answered without a Learn link", "warn"))

With monitoring set up as in [lab 13](../13-monitoring/README.md), open **Monitoring** in the
[AI Gateway portal](https://ai.gateway.azure.com) a few minutes later: **Models** counts the agent's identity apart
from you and your other apps, **Tools** shows the calls the toolbox made to `learn` as the project's identity, and
**Traces** shows one row per model or tool call. To chat with the agent in the Foundry portal, run
`azd ai agent show gateway-agent -e aigw-lab16-nb` in `infra/` and open its **Playground URL**.

### 9. Cap the agent's tokens

Give the alias a limit of 1,000 tokens per minute, counted per caller identity. A policy update replaces the alias's
whole policy array, so `set_policies` keeps the alias's other properties, and the cell keeps its other policies. It
saves the original array first, so the next step and **Clean up** can put it back exactly.

In [ ]:
alias_path = alias_resource["id"]
token_limit = {"type": "tokenLimit", "period": "minute", "count": 1000, "counterKey": ["identity"]}


def current_policies():
    policies = lab.arm("GET", alias_path)["properties"].get("policies") or []
    return [{k: v for k, v in p.items() if k != "scope"} for p in policies]   # the gateway adds scope on read


def set_policies(policies):
    """Replace the alias's whole policy array, and keep its other properties."""
    properties = {k: v for k, v in lab.arm("GET", alias_path)["properties"].items()
                  if k not in ("supportedEndpoints", "gatewayEndpoints", "provisioningState")}
    lab.arm("PUT", alias_path, {"properties": {**properties, "policies": policies}})
    return current_policies()


original_policies = [p for p in current_policies() if p != token_limit]   # drops a limit an earlier run left behind
applied = set_policies([p for p in original_policies if p["type"] != "tokenLimit"] + [token_limit])
display(brand.policies(alias, applied))

### 10. Trigger the limit, then remove it

The cell waits a minute for the policy to apply, then invokes the hosted agent up to four times, each in a new
conversation. The gateway counts tokens when a call completes, so the first invoke usually succeeds and uses well over
1,000 tokens, because Learn tool results are long. The next one fails, and the agent's logs show the gateway's `429`.

The counter belongs to the agent's identity: locally, the agent signs in as you, a caller with its own counter. The
cell then puts back the alias's original policies, even if an invoke fails in an unexpected way.

In [ ]:
prompt = "Summarize Azure API Management in three sentences with Learn links."
invokes, limit_error = [], None
try:
    brand.wait(60, "for the token limit to apply")
    for attempt in range(1, 5):
        exit_code, output, seconds = azd("ai", "agent", "invoke", "gateway-agent", prompt, "--new-conversation",
                                         title=f"Invoke {attempt}", check=False)
        answered = exit_code == 0 and bool(answer_of(output, "gateway-agent"))
        invokes.append((attempt, answered, seconds))
        if not answered:
            for _ in range(6):   # the logs can take a few seconds to arrive
                limit_error = next((message for code, message in reversed(gateway_errors()) if code == "429"), None)
                if limit_error:
                    break
                time.sleep(10)
            break
finally:
    restored = set_policies(original_policies)

display(brand.table("Invokes of the hosted agent", ["Invoke", "Result", "Time"], [
    (attempt, brand.chip("Answered", "ok") if answered else brand.chip("Failed", "bad"), f"{seconds} s")
    for attempt, answered, seconds in invokes]))
if limit_error:
    display(brand.note("The gateway answered the agent with 429", "bad", code=limit_error))
display(brand.note(f"Removed the token limit: {alias} has its original {len(restored)} policies again", "ok"))
assert limit_error, "No invoke hit the limit. Rerun steps 9 and 10."

### 11. Verify

The agent answered locally and in Foundry, without a model deployment and without a key: its model calls and its
tool calls went through the gateway with Microsoft Entra tokens, and the gateway enforced a limit on it without any
change to its code.

In [ ]:
deployments = lab.arm("GET", f"{account_id}/deployments", api_version=utils.COGNITIVE_API)["value"]
values = env_values()
key_like = [name for name in values if re.search(r"KEY|SECRET|PASSWORD", name)]
holders = [who for who, _, object_id, *_ in identities if has_gateway_user(object_id)]
results = [
    ("The local agent cites Microsoft Learn", "learn.microsoft.com" in local_answer, "azd ai agent invoke --local"),
    ("The hosted agent cites Microsoft Learn", "learn.microsoft.com" in hosted_answer, "azd ai agent invoke"),
    ("The Foundry resource has no model deployments", not deployments, f"{len(deployments)} deployments"),
    ("The azd environment holds no key", not key_like, ", ".join(key_like) or f"{len(values)} values, no key"),
    ("You, the project, and the agent have AI Gateway User", len(holders) == 3, f"{len(holders)} of 3"),
    ("The gateway capped the agent's tokens with 429", bool(limit_error), limit_error or "No 429"),
]
passed = sum(ok for _, ok, _ in results)
display(brand.table("Verify", ["Check", "Result", "Detail"], [
    (label, brand.chip("Pass", "ok") if ok else brand.chip("Fail", "bad"), detail) for label, ok, detail in results],
    badge=brand.chip(f"{passed} of {len(results)} passed", "ok" if passed == len(results) else "bad")))
assert passed == len(results), "Some checks failed: see the table above."

### 12. Clean up resources

The cell removes the roles of the agent and the project on the gateway first, while their identities still exist.
It removes your own AI Gateway User role only if this notebook assigned it. Then `azd down --purge` deletes the
Foundry resource group, with the connection, the toolbox, and the hosted agent, and purges the soft-deleted Foundry
resource, which takes a few minutes. Last, it removes the azd environment and the virtual environment that the local
run created. Your gateway, its alias, and the `learn` MCP server stay, because other labs reuse them.

In [ ]:
if current_policies() != original_policies:
    set_policies(original_policies)
    display(brand.note(f"Restored the original policies of {alias}", "ok"))

for who, shown, object_id, *_ in identities:
    if who == "You":   # only the role this notebook assigned to you
        path = assignment_path(object_id)
        mine = [path] if lab.arm("GET", path, api_version=utils.ROLES_API) else []
    else:              # every AI Gateway role of the agent and the project, at the gateway or below it
        mine = [a["id"] for a in lab.role_assignments(object_id)
                if a["scope"].lower().startswith(lab.gateway_id.lower())]
    if not mine:
        display(brand.note(f"{who}: no role assignment to remove", "neutral"))
    for assignment in mine:
        display(brand.note(f"{who}: removing AI Gateway User on the gateway", "brand"))
        lab.delete(assignment, api_version=utils.ROLES_API)

azd("down", "--purge", "--force", title="Delete the Foundry resources")
azd("env", "remove", env_name, "--force", title=f"Remove the azd environment {env_name}", environment=False)
if created_azure_folder:
    shutil.rmtree(infra / ".azure", ignore_errors=True)
if created_venv:
    shutil.rmtree(venv, ignore_errors=True)
log_path.unlink(missing_ok=True)

## Troubleshooting

- **`Run azd auth login`**: azd signs in separately from the Azure CLI. Run `azd auth login` in a terminal, with the
  same account and tenant as `az login`, then rerun step 0.
- **`AuthorizationFailed` on a role assignment**: you need **Owner**, **User Access Administrator**, or **Role Based
  Access Control Administrator** on the gateway. Contributor can't grant roles.
- **`azd provision` fails with a region or quota error**: set `location` in step 0 to another region with Foundry
  Hosted Agents, then rerun from step 1. Run **Clean up** first if the environment exists.
- **`Port 8088 is in use`**: another agent runs there, for example `azd ai agent run` from the guide. Stop it with
  **Ctrl+C** in its terminal, then rerun step 7.
- **The local run can't download packages, with a certificate error**: a proxy inspects TLS traffic. Set
  `UV_SYSTEM_CERTS=true` before you start Jupyter, then rerun step 7.
- **`403` that lasts more than 10 minutes**: check **Access** › **Entra ID** for the identity in the error: the agent
  in Foundry, you locally. Rerun step 3 or 6.
- **`HTTP 403 on a tool call through the toolbox`**: the toolbox reaches the `learn` server as the project's identity.
  Check its role in **Access** › **Entra ID**, and keep `HandshakeClient` in `main.py`: without it, the toolbox talks
  MCP `2026-07-28` to the gateway, which answers `403`.
- **`ran out of tokens per minute`**: the agent's calls need more tokens per minute than the model deployment behind
  the alias allows. The cell waits a minute and asks again; if it keeps failing, raise the deployment's capacity.
- **No invoke hit the limit**: the agent used fewer than 1,000 tokens per invoke. Rerun steps 9 and 10.
- **The kernel stopped during step 10**: the token limit may still be on the alias. Run **Clean up**, which restores
  the alias's original policies.

For the other errors in this lab, see [Troubleshooting](../TROUBLESHOOTING.md#lab-16-foundry-hosted-agent).

## Next steps

- Let a coding agent wire an app to the gateway in [lab 17](../17-coding-agent-plugin/README.md) and its notebook
  [17-coding-agent-plugin.ipynb](../17-coding-agent-plugin/17-coding-agent-plugin.ipynb).
- Scope roles to one alias or MCP server in [lab 11](../11-entra-rbac/README.md) and its notebook
  [11-entra-rbac.ipynb](../11-entra-rbac/11-entra-rbac.ipynb).
- Docs: [Connect Microsoft Foundry to AI Gateway](https://ai.gateway.azure.com/docs/foundry) and
  [Deploy a hosted agent](https://learn.microsoft.com/azure/foundry/agents/how-to/deploy-hosted-agent).
- Read this lab on the labs site: [https://azure.github.io/aigateway/labs/16-foundry-hosted-agent/](https://azure.github.io/aigateway/labs/16-foundry-hosted-agent/).